# Questão 1 - Logística de emergência

Integrantes: Eduardo - RM561474; João Abe - RM561446. Grupo 4. Seed 4, definida em `config.py`.

O grafo representa ruas de mão dupla. Um kit é uma unidade de carga equivalente, e cada ponto recebe sua demanda inteira ou fica sem atendimento. Comparamos os mesmos candidatos alcançáveis e a mesma capacidade. A distância influencia a heurística e o percurso; não faz parte da restrição da mochila.


In [ ]:
from pathlib import Path
import sys
import json

RAIZ = Path.cwd().resolve()
if not (RAIZ / 'src').is_dir():
    RAIZ = RAIZ.parent
if not (RAIZ / 'src').is_dir():
    raise RuntimeError('Abra o notebook dentro da pasta checkpoint4/notebooks.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from config import SEED, REPETICOES
print('Grupo e seed:', SEED)


Grupo e seed: 4


## Gerar os dados e executar os dois métodos
Esta célula recria os CSVs, resultados e três figuras. A capacidade pode ser alterada aqui.

In [ ]:
from src.logistica import executar_questao1, CAPACIDADE_PADRAO
resultado = executar_questao1(RAIZ, seed=SEED, capacidade=CAPACIDADE_PADRAO)
print('Vértices:', resultado['numero_vertices'])
print('Conexões:', resultado['numero_conexoes'])
print('Bloqueios:', resultado['vias_bloqueadas'])
print('Inacessíveis:', resultado['inacessiveis'])
for metodo in ['guloso', 'dp']:
    r = resultado[metodo]
    print(metodo, '-> benefício:', r['beneficio_total'],
          '| kits:', r['demanda_total'], '| locais:', r['selecionados'])
assert resultado['dp']['beneficio_total'] >= resultado['guloso']['beneficio_total']


Vértices: 21
Conexões: 44
Bloqueios: 5
Inacessíveis: ['P20']
guloso -> benefício: 2487 | kits: 80 | locais: ['P11', 'P07', 'P10', 'P04', 'P05', 'P08', 'P19', 'P15', 'P09', 'P01']
dp -> benefício: 2569 | kits: 80 | locais: ['P01', 'P03', 'P04', 'P05', 'P07', 'P09', 'P10', 'P11', 'P15', 'P19']


## Regra gulosa e rede

Usamos `prioridade * beneficio / (demanda * (1 + distancia_km))`. O numerador favorece urgência e benefício; o denominador penaliza carga e deslocamento. A cada parada, Dijkstra calcula distâncias pelas ruas disponíveis e a pontuação é recalculada. O `1` representa uma distância de referência de 1 km, evitando divisão por zero.

Isso explica a vantagem local, mas não garante a melhor combinação de atendimentos.

![Grafo, pesos e bloqueios](../figures/questao1/01_grafo.png)


## Programação dinâmica

- Estado: `DP[i][c]` guarda o maior benefício com as primeiras `i` regiões e até `c` kits.
- Decisão: incluir ou não a região atual.
- Base: `DP[0][c] = 0` e `DP[i][0] = 0`, pois as demandas são positivas.
- Se a demanda `d_i` ultrapassa `c`, copiamos `DP[i-1][c]`.
- Se ela cabe, usamos `max(DP[i-1][c], b_i + DP[i-1][c-d_i])`.
- Reconstrução: começamos em `DP[N][C]`. Quando o valor muda em relação à linha anterior, incluímos aquela região e descontamos sua demanda. Em empate, não incluímos a região atual.

![Atendimentos e percursos](../figures/questao1/02_solucoes.png)

![Evolução e reconstrução da DP](../figures/questao1/03_programacao_dinamica.png)


## Contraexemplo verificável

São três regiões: A precisa de 4 kits e vale 7; B e C precisam de 3 kits cada e valem 5 cada. Prioridades iguais a 1 e distâncias iguais a 1 km. A tem escore 0,875; B e C têm aproximadamente 0,833.

Com 4 kits, escolher A é ótimo. Com 6 kits, a regra continua escolhendo A, mas B+C vale 10 e cabe exatamente. Esse segundo caso prova que a escolha gulosa pode errar.


In [ ]:
from src.logistica import casos_pequenos
for caso in casos_pequenos():
    print('Capacidade:', caso['capacidade'])
    print('  Guloso:', caso['guloso']['selecionados'], caso['guloso']['beneficio_total'])
    print('  DP:    ', caso['dp']['selecionados'], caso['dp']['beneficio_total'])
assert casos_pequenos()[0]['guloso']['beneficio_total'] == 7
assert casos_pequenos()[1]['dp']['beneficio_total'] == 10


Capacidade: 4
  Guloso: ['A'] 7
  DP:     ['A'] 7
Capacidade: 6
  Guloso: ['A'] 7
  DP:     ['B', 'C'] 10


## Interpretação e custo

A DP maximiza benefício dentro do modelo da mochila. Os menores caminhos de cada trecho não tornam a rota completa ótima. Passar por um ponto no percurso também não significa entregar recursos nele.

A tabela usa tempo e espaço `O(NC)`. O guloso, com Dijkstra a cada parada, usa tempo `O(N(V+E) log V + N²)` e pode armazenar `O(NV)` vértices nos trechos da saída. Veja a justificativa detalhada em `docs/analise_complexidade.md`.
